# Option Instrument Tear Sheet

**Use after source analytics:** Reporting notebooks render results produced by the pricing, analytics, statement, and portfolio notebooks; they do not replace those calculation workflows.

**Purpose:** Render option valuation and Greeks in a compact instrument review page.

**Prerequisites:** `02_pricing/instruments/equity_and_options.ipynb`.

**What you'll learn:**

- Prepare option valuation and Greek inputs.
- Render `reporting.instrument_option_tearsheet`.
- Separate pricing model outputs from reviewer-facing layout.

This notebook renders a single-page HTML tear sheet for a **European equity call option**
using the polymorphic `reporting.instrument_tearsheet` API — pass an instrument JSON plus
a `MarketContext` and `as_of` date, and the library prices it with the recommended metric
set and renders the full report inline.

**Market setup:**
- A USD-OIS discount curve (4 knots, 2025-01-02 base).
- Spot price for `SPX-SPOT` inserted via `MarketContext.insert_price`.
- A dividend yield of 0 % (unitless scalar inserted under `SPX-DIVYIELD`).
- Implied volatility supplied via `pricing_overrides.implied_volatility = 0.20` — no full
  vol surface required.

**Pricing model:** `black76` — the standard Black-Scholes model for equity options.

The rendered sheet includes option-specific sections:
- **Key Metrics:** Notional, option premium (NPV).
- **Greeks:** Delta, Gamma, Vega, Theta, Rho, Charm.
- **Payoff at Expiry:** piecewise payoff diagram.

Tooltips are available natively in Jupyter. Open a saved `.html` file in a browser for
the richer crosshair + cursor-following tooltip.


In [ ]:
import datetime as dt
import json
from datetime import date

from finstack_quant.core.market_data import DiscountCurve, MarketContext
from finstack_quant.valuations.instruments.equity import EquityOption
from finstack_quant import reporting

In [ ]:
AS_OF = "2025-01-02"

# USD-OIS discount curve anchored to 2025-01-02.
ois = DiscountCurve(
    "USD-OIS",
    date(2025, 1, 2),
    [(0.0, 1.0), (0.25, 0.990), (0.5, 0.980), (1.0, 0.962), (2.0, 0.925)],
    day_count="act_365f",
)

mc = MarketContext().insert(ois)

# Insert SPX spot price and a zero dividend yield as scalar market items.
mc.insert_price("SPX-SPOT", 4500.0, "USD")
mc.insert_price("SPX-DIVYIELD", 0.0)  # unitless: 0 % continuous dividend yield

# Build a 1-year ATM SPX European call.
# Supply vol via pricing_overrides so a full vol surface is not required.
option = EquityOption(
    id="SPX-CALL-1Y",
    underlying_ticker="SPX",
    strike=4500.0,
    option_type="call",
    expiry="2026-01-02",
    notional={"amount": "100", "currency": "USD"},
    discount_curve_id="USD-OIS",
    spot_id="SPX-SPOT",
    vol_surface_id="SPX-VOL",   # referenced but overridden below
    div_yield_id="SPX-DIVYIELD",
    attributes={"tags": [], "meta": {}},
    pricing_overrides={"implied_volatility": 0.20},  # flat vol override: 20 %
)

option_json = json.loads(option.to_json())
print(f"Instrument type : {option_json['type']}")
print(f"Underlying      : {option_json['spec']['underlying_ticker']}")
print(f"Strike          : {option_json['spec']['strike']:,.0f}")
print(f"Option type     : {option_json['spec']['option_type']}")
print(f"Expiry          : {option_json['spec']['expiry']}")
print(f"Implied vol     : {option_json['spec']['pricing_overrides']['implied_volatility']:.0%}")

In [ ]:
# Render the full tear sheet in one call.
# instrument_tearsheet detects the instrument JSON, prices with the recommended
# equity-option metrics (delta, gamma, vega, theta, rho, charm) using model='black76',
# and renders the result inline via _repr_html_.
reporting.instrument_tearsheet(
    option_json,
    market=mc,
    as_of=AS_OF,
    model="black76",
    generated=dt.date(2026, 6, 19),
)

To save the tear sheet as a standalone HTML file:

```python
ts = reporting.instrument_tearsheet(option_json, market=mc, as_of=AS_OF, model="black76")
ts.save("option_tearsheet.html")
```

## Takeaways

- Reporting functions are presentation wrappers over analytics, valuation, statement, or portfolio results produced earlier in the curriculum.
- Keep the analytical source of truth in the typed objects or JSON specs, then render a tear sheet for review.
- Pass fixed `generated` dates in examples so notebook output remains reproducible.
